# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# k-Nearest Neighbors (kNN)

Classify by asking: what do my nearest neighbors look like?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

## Step 1: Load and Prepare Data

In [ ]:
# Load breast cancer data
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# **CRITICAL:** kNN requires feature scaling
# All features must be on same scale, else large-scale features dominate distance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")

## Step 2: How kNN Works

For a new patient's measurements:

1. Find k closest training patients (using scaled Euclidean distance)
2. Look at their diagnoses (benign or malignant)
3. Predict majority class

Example with k=5:
- 4 benign, 1 malignant among 5 nearest → predict **benign**

In [ ]:
# Train kNN with k=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"kNN (k=5) - Accuracy: {accuracy:.4f}")

## Step 3: Effect of k

- k=1: Each point just asks its single nearest neighbor (high variance, noisy)
- k=5: Majority vote among 5 (balanced)
- k=100: Majority vote among 100 (smoother, but ignores local patterns)

Trade-off: Small k overfits, large k underfits.

In [ ]:
# Try different k values
k_values = range(1, 31, 2)  # 1, 3, 5, ..., 29
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_score = knn.score(X_train_scaled, y_train)
    test_score = knn.score(X_test_scaled, y_test)
    
    train_scores.append(train_score)
    test_scores.append(test_score)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(k_values, train_scores, label='Training Accuracy', marker='o')
plt.plot(k_values, test_scores, label='Test Accuracy', marker='s')
plt.xlabel('k (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.title('kNN: Effect of k on Train/Test Performance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_k = k_values[np.argmax(test_scores)]
print(f"\nBest k: {best_k} (test accuracy: {max(test_scores):.4f})")

## Step 4: Distance Metrics

In [ ]:
# Different distance metrics
metrics = ['euclidean', 'manhattan', 'minkowski']

print(f"{'Metric':<15} {'Accuracy':>10}")
print("-" * 25)

for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_train_scaled, y_train)
    score = knn.score(X_test_scaled, y_test)
    print(f"{metric:<15} {score:>10.4f}")

## Step 5: Importance of Scaling

In [ ]:
# Show why scaling matters
# Train WITHOUT scaling
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)  # Use unscaled data
score_unscaled = knn_unscaled.score(X_test, y_test)

# Train WITH scaling
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)  # Use scaled data
score_scaled = knn_scaled.score(X_test_scaled, y_test)

print(f"Without scaling: {score_unscaled:.4f}")
print(f"With scaling:    {score_scaled:.4f}")
print(f"\nDifference: {(score_scaled - score_unscaled):.4f}")
print("\nWhy? Without scaling, large-magnitude features dominate distance calculation.")

## Step 6: Cross-Validation

In [ ]:
# More robust k selection using cross-validation
best_score = 0
best_k = 1

for k in range(1, 31):
    knn = KNeighborsClassifier(n_neighbors=k)
    cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    mean_score = cv_scores.mean()
    
    if mean_score > best_score:
        best_score = mean_score
        best_k = k

print(f"Best k by cross-validation: {best_k}")
print(f"Mean CV accuracy: {best_score:.4f}")